# MEDISCOPE — 02 Data Preprocessing

## Cleaning, validation and reproducible preparation

This notebook expands the original preprocessing notebook into an auditable account of the cleaning stage used before feature engineering.

### Objectives

- preserve the original research extract;
- validate the dataset structure;
- standardise date fields;
- clean implausible `Age at ART Initiation` values;
- quantify missing/invalid dates;
- check duplicates and data types;
- save a reproducible intermediate Parquet artefact.

The production implementation lives under `src/preprocessing.py`. This notebook documents the same methodological decisions in an assessor-friendly form.

## 1. Project setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


def find_project_root(start: Path | None = None) -> Path:
    """Locate the MEDISCOPE repository root from common notebook launch locations."""
    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (
            (candidate / "src").is_dir()
            and (candidate / "api").is_dir()
            and (candidate / "requirements.txt").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Unable to locate the MEDISCOPE repository root. "
        "Run this notebook from the repository or notebooks directory."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = PROJECT_ROOT / "models" / "trained"
REPORT_DIR = PROJECT_ROOT / "reports" / "evaluation"

print(f"Project root: {PROJECT_ROOT}")

## 2. Load the immutable raw dataset

In [ ]:
RAW_FILE = RAW_DIR / "LTFU in HIV DataSet NDR.xlsx"
OUTPUT_FILE = PROCESSED_DIR / "01_dates_converted.parquet"

if not RAW_FILE.exists():
    raise FileNotFoundError(f"Raw dataset not found: {RAW_FILE}")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_excel(RAW_FILE)
df = df_raw.copy()

print(f"Loaded: {RAW_FILE.name}")
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")

The copy ensures that preprocessing operations cannot alter the in-memory representation used to describe the untouched source.

## 3. Baseline quality audit

In [ ]:
baseline = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": df.isna().mean().mul(100),
    "unique": df.nunique(dropna=True),
})

print(f"Exact duplicate rows: {df.duplicated().sum():,}")
baseline.sort_values("missing_pct", ascending=False).head(30)

## 4. Convert date fields

Date fields are converted using `errors="coerce"` so malformed/unparseable values become `NaT` and can be measured explicitly.

This is preferable to silently accepting inconsistent text dates because later features such as age, treatment duration and recency depend on valid temporal ordering.

In [ ]:
DATE_COLUMNS = [
    "Date Of Birth",
    "ART Start Date",
    "Last Drug Pickup date",
    "Last Drug Pickup date Q1",
    "Last Drug Pickup date Q2",
    "Last Drug Pickup date Q3",
    "Last Drug Pickup date Q4",
    "Last Clinic Visit Date",
    "Date Of Current Viral Load",
    "Date Of Current Viral Load Q1",
    "Date Of Current Viral Load Q2",
    "Date Of Current Viral Load Q3",
    "Date Of Current Viral Load Q4",
    "Patient Deceased Date",
    "Transferred Out Date",
    "Transferred In Date",
]

date_columns_present = [
    column for column in DATE_COLUMNS
    if column in df.columns
]

for column in date_columns_present:
    df[column] = pd.to_datetime(
        df[column],
        errors="coerce",
    )

print(f"Converted {len(date_columns_present)} date fields.")

## 5. Clean `Age at ART Initiation`

The validated preprocessing run identified two clear plausibility problems:

- **50 negative ages**;
- **2 ages above 100 years**.

These values were converted to missing rather than inventing replacements. This preserves the fact that the original values were invalid while allowing later model preprocessing to handle missingness consistently.

In [ ]:
AGE_COLUMN = "Age at ART Initiation"

age_cleaning_summary = {}

if AGE_COLUMN in df.columns:
    age = pd.to_numeric(df[AGE_COLUMN], errors="coerce")

    negative_mask = age < 0
    over_100_mask = age > 100

    age_cleaning_summary["negative_to_missing"] = int(negative_mask.sum())
    age_cleaning_summary["over_100_to_missing"] = int(over_100_mask.sum())

    age = age.mask(negative_mask | over_100_mask)
    df[AGE_COLUMN] = age

    age_cleaning_summary["valid_numeric_remaining"] = int(age.notna().sum())
    age_cleaning_summary["missing_or_invalid_after_cleaning"] = int(age.isna().sum())

pd.Series(age_cleaning_summary, name="records").to_frame()

For the validated source extract, the expected result is:

| Check | Records |
|---|---:|
| Negative ages converted to missing | 50 |
| Ages above 100 converted to missing | 2 |
| Valid numeric ages remaining | 304,197 |
| Missing / invalid after cleaning | 76 |

## 6. Date completeness validation

In [ ]:
date_validation = pd.DataFrame({
    "Column": date_columns_present,
    "Missing / Invalid Dates": [
        int(df[column].isna().sum())
        for column in date_columns_present
    ],
    "Percentage (%)": [
        float(df[column].isna().mean() * 100)
        for column in date_columns_present
    ],
}).sort_values("Percentage (%)", ascending=False)

date_validation

The successful preprocessing run identified, among other observations:

- `Last Drug Pickup date`: approximately **4.29%** missing/invalid;
- `Last Clinic Visit Date`: approximately **3.58%**;
- `Date Of Current Viral Load`: approximately **21.74%**;
- Q1/Q2/Q4 viral-load dates: approximately **47–54%** missing;
- Q3 pickup and viral-load dates: **100% missing** in the source extract;
- deceased/transfer dates are mostly absent, which is expected for many active/non-transfer records.

These fields are therefore not treated as uniformly complete measurements.

## 7. Validate numeric fields without destructive imputation

At this stage, missing clinical values are **not globally filled with arbitrary constants**. Model-specific pipelines can later perform appropriate imputation while preserving missingness indicators where designed.

In [ ]:
candidate_numeric = [
    "Age at ART Initiation",
    "Current Age",
    "Days Of ARV Refill",
    "Current Viral Load",
]

numeric_validation = {}

for column in candidate_numeric:
    if column in df.columns:
        values = pd.to_numeric(df[column], errors="coerce")
        numeric_validation[column] = {
            "valid_numeric": int(values.notna().sum()),
            "missing_or_invalid": int(values.isna().sum()),
            "minimum": values.min(),
            "median": values.median(),
            "maximum": values.max(),
        }

pd.DataFrame(numeric_validation).T

## 8. Verify transformation boundaries

Preprocessing should not yet create the final target or encoded feature matrix. Those transformations belong to the feature-engineering stage, keeping responsibilities separated and easier to audit.

In [ ]:
print("Rows preserved:", len(df) == len(df_raw))
print("Columns preserved:", set(df.columns) == set(df_raw.columns))
print("Raw dataframe object unchanged:", df is not df_raw)

## 9. Save the intermediate Parquet dataset

Parquet preserves typed columns more reliably and compactly than CSV for this stage.

In [ ]:
df.to_parquet(
    OUTPUT_FILE,
    index=False,
)

print("Dataset successfully saved.")
print(f"Location: {OUTPUT_FILE}")
print(f"File exists: {OUTPUT_FILE.exists()}")
print(f"File size: {OUTPUT_FILE.stat().st_size / (1024**2):,.2f} MB")

## 10. Reload and verify the saved artefact

In [ ]:
reloaded = pd.read_parquet(OUTPUT_FILE)

assert len(reloaded) == len(df)
assert list(reloaded.columns) == list(df.columns)

print(f"Reload verification passed: {len(reloaded):,} rows.")

## 11. Reproduce with the production module (optional)

The repository's production preprocessing implementation remains the authoritative executable pipeline. Run it from the project root when you intentionally want to regenerate the processed artefact:

```powershell
python -m src.preprocessing
```

Keeping the production logic in `src/` prevents the notebook from becoming the only place where important cleaning rules exist.

## Key findings

- The source dataset contains **304,273 records**.
- Dates require explicit conversion and completeness reporting.
- `Age at ART Initiation` contained a small number of clearly implausible values that were converted to missing.
- Missingness varies considerably across longitudinal clinical fields.
- No broad destructive imputation is performed at this stage.
- The cleaned date-aware intermediate dataset is persisted as `data/processed/01_dates_converted.parquet`.

### Next notebook

`03_feature_engineering.ipynb` examines how the cleaned clinical data is transformed into the final modelling representation.